# DP-SGD UCI Adult demo

Demo gọn cho thuyết trình: load dữ liệu Adult, train baseline, train DP-SGD với `noise_multiplier = 1.5`, in bảng so sánh và vẽ các biểu đồ tradeoff từ kết quả đã báo cáo.


In [ ]:
import importlib.util
import sys
import subprocess

if importlib.util.find_spec("opacus") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "opacus"])


In [ ]:
from collections.abc import Sequence
from pathlib import Path
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from opacus import PrivacyEngine
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = Path("adult")
BATCH_SIZE = 256
EPOCHS = 20
BASELINE_LR = 1e-3
DP_LR = 0.05
DELTA = 1e-5
NOISE_MULTIPLIER = 1.5
MAX_GRAD_NORM = 1.0
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)


In [ ]:
columns = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week", "native_country", "income",
]
numeric_cols = ["age", "fnlwgt", "education_num", "capital_gain", "capital_loss", "hours_per_week"]


def load_adult_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, names=columns, skipinitialspace=True, na_values="?", comment="|")
    df["income"] = df["income"].str.replace(".", "", regex=False)
    return df.dropna().reset_index(drop=True)


train_df = load_adult_csv(DATA_DIR / "adult.data")
test_df = load_adult_csv(DATA_DIR / "adult.test")

x_train_raw = train_df.drop(columns="income")
x_test_raw = test_df.drop(columns="income")
y_train = (train_df["income"] == ">50K").astype("float32")
y_test = (test_df["income"] == ">50K").astype("float32")

combined = pd.concat([x_train_raw, x_test_raw], axis=0)
combined = pd.get_dummies(
    combined,
    columns=[col for col in combined.columns if col not in numeric_cols],
    dtype="float32",
)
x_train = combined.iloc[: len(train_df)].copy()
x_test = combined.iloc[len(train_df) :].copy()

means = x_train[numeric_cols].mean()
stds = x_train[numeric_cols].std().replace(0, 1)
x_train[numeric_cols] = (x_train[numeric_cols] - means) / stds
x_test[numeric_cols] = (x_test[numeric_cols] - means) / stds

x_train_tensor = torch.tensor(x_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values.reshape(-1, 1), dtype=torch.float32)
x_test_tensor = torch.tensor(x_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values.reshape(-1, 1), dtype=torch.float32)

train_loader = DataLoader(TensorDataset(x_train_tensor, y_train_tensor), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TensorDataset(x_test_tensor, y_test_tensor), batch_size=BATCH_SIZE, shuffle=False)
print("Input features:", x_train_tensor.shape[1])


In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dims: Sequence[int] = (128, 64, 32)) -> None:
        super().__init__()
        layers: list[nn.Module] = []
        previous_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.extend([nn.Linear(previous_dim, hidden_dim), nn.ReLU()])
            previous_dim = hidden_dim
        layers.append(nn.Linear(previous_dim, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


criterion = nn.BCEWithLogitsLoss()


def evaluate(model: nn.Module) -> dict[str, float]:
    model.eval()
    correct = total = tp = fp = fn = 0
    with torch.no_grad():
        for features, labels in test_loader:
            features = features.to(DEVICE)
            labels = labels.to(DEVICE)
            preds = (torch.sigmoid(model(features)) >= 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.numel()
            tp += ((preds == 1) & (labels == 1)).sum().item()
            fp += ((preds == 1) & (labels == 0)).sum().item()
            fn += ((preds == 0) & (labels == 1)).sum().item()
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    return {"accuracy": correct / total, "precision": precision, "recall": recall, "f1_score": f1}


In [ ]:
def train_baseline() -> dict[str, float]:
    torch.manual_seed(RANDOM_SEED)
    model = MLP(x_train_tensor.shape[1]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=BASELINE_LR)
    start = time.time()
    for epoch in range(EPOCHS):
        model.train()
        for features, labels in train_loader:
            features = features.to(DEVICE)
            labels = labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(features), labels)
            loss.backward()
            optimizer.step()
    return {**evaluate(model), "training_time": time.time() - start}


baseline_metrics = train_baseline()
print("Baseline accuracy:", round(baseline_metrics["accuracy"], 4))
print("Baseline F1-score:", round(baseline_metrics["f1_score"], 4))


In [ ]:
def train_dp_sgd() -> dict[str, float]:
    torch.manual_seed(RANDOM_SEED)
    model = MLP(x_train_tensor.shape[1]).to(DEVICE)
    optimizer = torch.optim.SGD(model.parameters(), lr=DP_LR, momentum=0.9)
    dp_train_loader = DataLoader(
        TensorDataset(x_train_tensor, y_train_tensor),
        batch_size=BATCH_SIZE,
        shuffle=True,
    )
    privacy_engine = PrivacyEngine(accountant="prv")
    model, optimizer, private_loader = privacy_engine.make_private(
        module=model,
        optimizer=optimizer,
        data_loader=dp_train_loader,
        noise_multiplier=NOISE_MULTIPLIER,
        max_grad_norm=MAX_GRAD_NORM,
    )
    start = time.time()
    for epoch in range(EPOCHS):
        model.train()
        for features, labels in private_loader:
            features = features.to(DEVICE)
            labels = labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(features), labels)
            loss.backward()
            optimizer.step()
    return {
        **evaluate(model),
        "epsilon": privacy_engine.get_epsilon(DELTA),
        "training_time": time.time() - start,
    }


dp_metrics = train_dp_sgd()
print("DP-SGD epsilon:", round(dp_metrics["epsilon"], 4))
print("DP-SGD accuracy:", round(dp_metrics["accuracy"], 4))
print("DP-SGD F1-score:", round(dp_metrics["f1_score"], 4))
print("Accuracy drop:", round(baseline_metrics["accuracy"] - dp_metrics["accuracy"], 4))


In [ ]:
comparison = pd.DataFrame(
    [
        {"method": "Baseline", "epsilon": None, **baseline_metrics},
        {"method": "DP-SGD noise=1.5", **dp_metrics},
    ]
)
display(comparison)


In [ ]:
reported_noise_results = pd.DataFrame(
    [
        {"noise_multiplier": 0.5, "epsilon": 15.5548, "accuracy": 0.8491},
        {"noise_multiplier": 0.8, "epsilon": 3.5038, "accuracy": 0.8485},
        {"noise_multiplier": 1.0, "epsilon": 2.1115, "accuracy": 0.8479},
        {"noise_multiplier": 1.5, "epsilon": 1.0946, "accuracy": 0.8464},
        {"noise_multiplier": 2.0, "epsilon": 0.7501, "accuracy": 0.8460},
        {"noise_multiplier": 3.0, "epsilon": 0.4637, "accuracy": 0.8425},
    ]
)

plt.figure(figsize=(7, 4))
plt.plot(reported_noise_results["epsilon"], reported_noise_results["accuracy"], marker="o")
plt.xlabel("Epsilon")
plt.ylabel("Accuracy")
plt.title("Privacy Utility Tradeoff")
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(reported_noise_results["noise_multiplier"], reported_noise_results["epsilon"], marker="o")
plt.xlabel("Noise multiplier")
plt.ylabel("Epsilon")
plt.title("Noise Multiplier vs Epsilon")
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(reported_noise_results["noise_multiplier"], reported_noise_results["accuracy"], marker="o")
plt.xlabel("Noise multiplier")
plt.ylabel("Accuracy")
plt.title("Noise Multiplier vs Accuracy")
plt.grid(True)
plt.tight_layout()
plt.show()
